# Inter-Annotator Agreement

This notebook computes inter-annotator agreement scores (Cohen's kappa and character error rate) between two annotators for the coordinate extraction task on entries from the EDDA-Coordinata dataset.

In [ ]:
import json
from Levenshtein import distance
import pandas as pd

## Load datasets

In [3]:
# Load datasets
with open("../data/annotator1.json", "r", encoding="utf-8") as f1, open("../data/annotator2.json", "r", encoding="utf-8") as f2:
    data_annotator1 = json.load(f1)
    data_annotator2 = json.load(f2)

print("Number of entries :\n Annotator 1 : {}\n Annotator 2 : {}".format(len(data_annotator1), len(data_annotator2)))

# remove entries without coordinates (if the coord_extr key exists)
data_annotator1 = [entry for entry in data_annotator1 if "coord_extr" in entry]
data_annotator2 = [entry for entry in data_annotator2 if "coord_extr" in entry]

print("Number of entries with coordinates :\n Annotator 1 : {}\n Annotator 2 : {}".format(len(data_annotator1), len(data_annotator2)))



Number of entries :
 Annotator 1 : 15274
 Annotator 2 : 15510
Number of entries with coordinates :
 Annotator 1 : 4779
 Annotator 2 : 4650


## Points

### Preprocessing

In [4]:

#  remove entries with multiple coordinates
points1 = [entry for entry in data_annotator1 if entry["coord_extr"] is not None and len(entry["coord_extr"]) == 1 and len(entry["coord_extr"][0]) == 1]
points2 = [entry for entry in data_annotator2 if entry["type"] is not None and entry["type"] == "Simple" and  entry["subtype"] == "Point"]

print("Number of entries with a single coordinate (Point) :\n Annotator 1 : {}\n Annotator 2 : {}".format(len(points1), len(points2)))

# remove entries with only one latitude or one longitude (N + E or N + O) to keep well-formed coordinates
points1 = [entry for entry in points1 if ("N" in entry["coord_extr"][0][0] or "S" in entry["coord_extr"][0][0]) and ("E" in entry["coord_extr"][0][0] or "W" in entry["coord_extr"][0][0])]
points2 = [entry for entry in points2 if ("N" in entry["coord_extr"][0][0] or "S" in entry["coord_extr"][0][0]) and ("E" in entry["coord_extr"][0][0] or "W" in entry["coord_extr"][0][0])]

print("Number of entries with a single well-formed coordinate (Point) :\n Annotator 1 : {}\n Annotator 2 : {}".format(len(points1), len(points2)))


Number of entries with a single coordinate (Point) :
 Annotator 1 : 4508
 Annotator 2 : 4505
Number of entries with a single well-formed coordinate (Point) :
 Annotator 1 : 4272
 Annotator 2 : 4331


### Comparison of both annotations

In [5]:
# set comparison key
key = "entreeid"

# build id mappings
ids1 = {entry[key]: entry for entry in points1}
ids2 = {entry[key]: entry for entry in points2}

# intersections and differences
commons = set(ids1.keys()) & set(ids2.keys())
uniques_annotator1 = set(ids1.keys()) - set(ids2.keys())
uniques_annotator2 = set(ids2.keys()) - set(ids1.keys())

print(f"Common entries : {len(commons)}")
print(f"Unique entries in annotator 1 : {len(uniques_annotator1)}")
print(f"Unique entries in annotator 2 : {len(uniques_annotator2)}")

commons_entries = [entry for entry in points1 if entry[key] in commons]
uniques_annotator1_entries = [entry for entry in points1 if entry[key] in uniques_annotator1]
uniques_annotator2_entries = [entry for entry in points2 if entry[key] in uniques_annotator2]

same_coords = []
diff_coords = []
diff_coords_annotator1 = []
diff_coords_annotator2 = []
coords_annotator1 = []
coords_annotator2 = []

for eid in commons:
    coords1 = ids1[eid].get("coord_extr")
    coords_annotator1.append(coords1[0][0])
    coords2 = ids2[eid].get("coord_extr")
    coords_annotator2.append(coords2[0][0])
    if coords1 == coords2:
        same_coords.append({
            "entreeid": eid,
            "vedette": ids1[eid].get("vedette"),
            "coord_extr": coords1
        })
    else:
        diff_coords_annotator1.append(coords1[0][0])
        diff_coords_annotator2.append(coords2[0][0])

        diff_coords.append({
            "entreeid": eid,
            "vedette": ids1[eid].get("vedette"),
            "texte": ids1[eid].get("texte"),
            "coord_extr_geode": coords1,
            "coord_extr_nugues": coords2
        })

print(f"Common entries with same coordinates : {len(same_coords)}")
print(f"Common entries with different coordinates : {len(diff_coords)}")
#agreement rate  (nombre d’accord exact / nombre d’annotations) 
print(f"Agreement rate : {len(same_coords) / len(commons):.3f}")


def calculate_cer(reference, hypothesis):
    if len(reference) == 0:
        return float('inf')  # or define a convention
    edit_distance = distance(reference, hypothesis)
    return edit_distance / len(reference)

def average_cer(references, hypotheses):
    assert len(references) == len(hypotheses), "Mismatched list lengths."
    total_edits = 0
    total_chars = 0
    for ref, hyp in zip(references, hypotheses):
        total_edits += distance(ref, hyp)
        total_chars += len(ref)
    return total_edits / total_chars if total_chars > 0 else float('inf')
# Calculate CER for all common entries
cer = average_cer(coords_annotator1, coords_annotator2)
print(f"Micro-average Character Error Rate (CER): {cer:.3f}")  

cer = average_cer(diff_coords_annotator1, diff_coords_annotator2)
print(f"Micro-average CER without exact matches: {cer:.3f}")  



Common entries : 4221
Unique entries in annotator 1 : 51
Unique entries in annotator 2 : 110
Common entries with same coordinates : 4140
Common entries with different coordinates : 81
Agreement rate : 0.981
Micro-average Character Error Rate (CER): 0.004
Micro-average CER without exact matches: 0.185


## Surfaces

### Preprocessing

In [6]:
path = "../data/"
df_surfaces_diff = pd.read_excel(path + "surfaces_diff.xlsx")
df_surfaces_annotator1 = pd.read_excel(path + "surfaces_annotator1.xlsx")

In [7]:
df_surfaces_diff = df_surfaces_diff[["entreeid", "vedette", "texte", "Coord_n", "Coord_g", "gold"]]
df_surfaces_diff = df_surfaces_diff.rename(columns={"Coord_n": "annotator1", "Coord_g": "annotator2"})
df_surfaces_annotator1 = df_surfaces_annotator1[["entreeid", "vedette", "texte", "n", "gold"]]
df_surfaces_annotator1 = df_surfaces_annotator1.rename(columns={"n": "annotator1"})

### Comparison of both annotations

Membership agreement rate = proportion of times b ∈ a

In [8]:
df_merge = pd.merge(df_surfaces_diff, df_surfaces_annotator1, on=["entreeid", "vedette", "texte", "annotator1"], how="outer")
df_merge


,entreeid,vedette,texte,annotator1,annotator2,gold_x,gold_y
0,v1-1046-0,ALBANIE,"*​ ALBANIE, (Geog.)​ province de la Turquie Eu...","[[""39 N 36 18' E"", ""43 30' N 39 40' E""]]","[[""39 N 36 18' E"", ""43 30' N 39 40' E""]]",NaN,n
1,v1-1264-0,ALLEMAGNE,"*​ ALLEMAGNE, (Geog.)​ grand pays situé au mil...","[['46 N 23 E', '55 N 37 E']]","[['46 N 23 E', '55 N 37 E']]",NaN,n
2,v1-131-0,ABISSINIE,"*​ ABISSINIE, s. f. grand Pays & Royaume d’Afr...","[['6 N 48 E', '20 N 65 E']]","[['6 N 48 E', '20 N 65 E']]",NaN,n
3,v1-135-0,ABLAI,"*​ ABLAI, s. contrée de la grande Tartarie. Lo...","[['51 N 91 E', '54 N 101 E']]","[['51 N 91 E', '54 N 101 E']]",NaN,n
4,v1-1359-0,ALSACE,"*​ ALSACE, province de France, bornée à l’est ...","[[""47 36' N 24 30' E"", ""49 N 35 20' E""]]","[[""47 36' N 24 30' E"", ""49 N 35 20' E""]]",NaN,p
...,...,...,...,...,...,...,...
140,v9-504-0,KOPING,"KOPING, (Géog.)​ Kopingia, ville de Suede dans...","[['59 N 36 E', '60 N 37 E']]","[['59 N 36 E', '60 N 37 E']]",NaN,n
141,v9-609-0,LABRADOR,"LABRADOR​, ​Estotilandia​, (Géog.)​ grand pays...","[['50 N 301 E', '63 N 323 E']]","[['50 N 301 E', '63 N 323 E']]",NaN,n
142,v9-662-0,"LADOGA, lac","LADOGA, lac, (Géogr.)​ grand lac de l’empire R...","[[""51 60' N 41 39' E"", ""60 N 51 20' E""]]",NaN,n,n
143,v9-738-0,LALAND,"LALAND, Lalandia, (Géog.)​ petite île du royau...","[[""54 48' N 29 20' E"", ""54 53' N 29 55' E""]]","[[""54 48' N 29 20' E"", '53 N 55 E']]",n,n


In [9]:
commons = df_merge[(df_merge["annotator1"].isna() == False) & (df_merge["annotator2"].isna() == False)]
commons

,entreeid,vedette,texte,annotator1,annotator2,gold_x,gold_y
0,v1-1046-0,ALBANIE,"*​ ALBANIE, (Geog.)​ province de la Turquie Eu...","[[""39 N 36 18' E"", ""43 30' N 39 40' E""]]","[[""39 N 36 18' E"", ""43 30' N 39 40' E""]]",NaN,n
1,v1-1264-0,ALLEMAGNE,"*​ ALLEMAGNE, (Geog.)​ grand pays situé au mil...","[['46 N 23 E', '55 N 37 E']]","[['46 N 23 E', '55 N 37 E']]",NaN,n
2,v1-131-0,ABISSINIE,"*​ ABISSINIE, s. f. grand Pays & Royaume d’Afr...","[['6 N 48 E', '20 N 65 E']]","[['6 N 48 E', '20 N 65 E']]",NaN,n
3,v1-135-0,ABLAI,"*​ ABLAI, s. contrée de la grande Tartarie. Lo...","[['51 N 91 E', '54 N 101 E']]","[['51 N 91 E', '54 N 101 E']]",NaN,n
4,v1-1359-0,ALSACE,"*​ ALSACE, province de France, bornée à l’est ...","[[""47 36' N 24 30' E"", ""49 N 35 20' E""]]","[[""47 36' N 24 30' E"", ""49 N 35 20' E""]]",NaN,p
...,...,...,...,...,...,...,...
138,v9-2313-0,MADAGASCAR,"MADAGASCAR, (Géogr.)​ île immense sur les côte...","[['12 12\' S 62 1\' 15"" E', '25 10\' S 62 1\'...","[['12 12 N 62 1 15E', '25 10 N E']]",n,n
139,v9-2346-0,MAELSTROM,"MAELSTROM, (Géogr.)​ espece de goufre de l’Océ...","[[""68 10' N 28 E"", ""68 15' N 28 E""]]","[[""68 10' N 28 E"", ""68 15' N E']]",n,n
140,v9-504-0,KOPING,"KOPING, (Géog.)​ Kopingia, ville de Suede dans...","[['59 N 36 E', '60 N 37 E']]","[['59 N 36 E', '60 N 37 E']]",NaN,n
141,v9-609-0,LABRADOR,"LABRADOR​, ​Estotilandia​, (Géog.)​ grand pays...","[['50 N 301 E', '63 N 323 E']]","[['50 N 301 E', '63 N 323 E']]",NaN,n


In [10]:
# how many rows have the same value in annotator1 and annotator2
same = commons[commons["annotator1"] == commons["annotator2"]]
len(same)

46

In [11]:
# count rows where annotator1 and annotator2 are both not equal to NaN
commons = df_merge[(df_merge["annotator1"].isna() == False) & (df_merge["annotator2"].isna() == False)]
uniques_annotator1 = df_merge[(df_merge["annotator1"].isna() == False) & (df_merge["annotator2"].isna() == True)]
uniques_annotator2 = df_merge[(df_merge["annotator1"].isna() == True) & (df_merge["annotator2"].isna() == False)]
same_coords = commons[commons["annotator1"] == commons["annotator2"]]
diff_coords = commons[commons["annotator1"] != commons["annotator2"]]

print(f"Number of surfaces : {len(df_merge)}")
print(f"Common entries : {len(commons)}")
print(f"Unique entries in annotator 1 : {len(uniques_annotator1)}")
print(f"Unique entries in annotator 2 : {len(uniques_annotator2)}")


print(f"Common entries with same coordinates : {len(same_coords)}")
print(f"Common entries with different coordinates : {len(diff_coords)}")

print(f"Agreement rate : {len(same_coords) / len(commons):.3f}")

# Calculate CER for all common entries
cer = average_cer(commons["annotator1"], commons["annotator2"])
print(f"Micro-average Character Error Rate (CER): {cer:.3f}")  

cer = average_cer(diff_coords["annotator1"], diff_coords["annotator2"])
print(f"Micro-average CER without exact matches: {cer:.3f}")  



Number of surfaces : 145
Common entries : 88
Unique entries in annotator 1 : 52
Unique entries in annotator 2 : 5
Common entries with same coordinates : 46
Common entries with different coordinates : 42
Agreement rate : 0.523
Micro-average Character Error Rate (CER): 0.105
Micro-average CER without exact matches: 0.209


In [12]:
df_merge

,entreeid,vedette,texte,annotator1,annotator2,gold_x,gold_y
0,v1-1046-0,ALBANIE,"*​ ALBANIE, (Geog.)​ province de la Turquie Eu...","[[""39 N 36 18' E"", ""43 30' N 39 40' E""]]","[[""39 N 36 18' E"", ""43 30' N 39 40' E""]]",NaN,n
1,v1-1264-0,ALLEMAGNE,"*​ ALLEMAGNE, (Geog.)​ grand pays situé au mil...","[['46 N 23 E', '55 N 37 E']]","[['46 N 23 E', '55 N 37 E']]",NaN,n
2,v1-131-0,ABISSINIE,"*​ ABISSINIE, s. f. grand Pays & Royaume d’Afr...","[['6 N 48 E', '20 N 65 E']]","[['6 N 48 E', '20 N 65 E']]",NaN,n
3,v1-135-0,ABLAI,"*​ ABLAI, s. contrée de la grande Tartarie. Lo...","[['51 N 91 E', '54 N 101 E']]","[['51 N 91 E', '54 N 101 E']]",NaN,n
4,v1-1359-0,ALSACE,"*​ ALSACE, province de France, bornée à l’est ...","[[""47 36' N 24 30' E"", ""49 N 35 20' E""]]","[[""47 36' N 24 30' E"", ""49 N 35 20' E""]]",NaN,p
...,...,...,...,...,...,...,...
140,v9-504-0,KOPING,"KOPING, (Géog.)​ Kopingia, ville de Suede dans...","[['59 N 36 E', '60 N 37 E']]","[['59 N 36 E', '60 N 37 E']]",NaN,n
141,v9-609-0,LABRADOR,"LABRADOR​, ​Estotilandia​, (Géog.)​ grand pays...","[['50 N 301 E', '63 N 323 E']]","[['50 N 301 E', '63 N 323 E']]",NaN,n
142,v9-662-0,"LADOGA, lac","LADOGA, lac, (Géogr.)​ grand lac de l’empire R...","[[""51 60' N 41 39' E"", ""60 N 51 20' E""]]",NaN,n,n
143,v9-738-0,LALAND,"LALAND, Lalandia, (Géog.)​ petite île du royau...","[[""54 48' N 29 20' E"", ""54 53' N 29 55' E""]]","[[""54 48' N 29 20' E"", '53 N 55 E']]",n,n


In [13]:
df_merge[(df_merge["gold_y"] == 'n') | (df_merge["gold_y"] == 'p') & (df_merge["gold_y"].isna() == False)]

corrections = df_merge[(df_merge["gold_x"] == 'corr') | (df_merge["gold_y"] != 'n') & (df_merge["gold_y"] != 'p') & (df_merge["gold_y"].isna() == False)]
print(f"New corrections : {len(corrections)}")

New corrections : 14


In [14]:
diff_coords

,entreeid,vedette,texte,annotator1,annotator2,gold_x,gold_y
16,v1-447-0,AÇORES,"*​ AÇORES, s. Isles de l’Amérique qui appartie...","[['39 N 346 E', '39 N 354 E']]","[['39 N 346 E', ' N 354 E']]",n,n
20,v10-1019-11,Mer Adriatique,"Mer Adriatique, (Géog.)​ Adriaticum mare ; ce ...","[['40 N', ""45 25' N""]]","[['40 N E', ""45 25' N E""]]",n,n
21,v10-1180-0,METELIN,"METELIN, (Géog.)​ île considérable de l’Archip...","[[""39 15' N 43 52' E"", ""39 15' N 44 31' E""]]","[[""39 15' N 43 52' E"", "" N 44 31' E""]]",n,n
22,v10-1297-0,MICHIGAN,"MICHIGAN, (Géog.)​ grand lac de l’Amérique sep...","[[""41 45' N"", ""49 30' N""]]","[[""41 45' N E"", ""49 30' N E""]]",n,n
32,v11-1455-0,ONÉGA lac d’,ONÉGA lac d’. (Géogr.)​ grand lac de l’empire ...,"[[""60 46' N 53 E"", '63 N 64 E']]","[[""60 46' N 53 E"", ""63 N 64 E""]]",NaN,n
33,v11-1751-0,ORIXA,"ORIXA, (Géog.)​ royaume de l’Indoustan, sur le...","[[""98 20' E"", ""102 20' E""]]","[["" N 98 20' E"", ""N 102 20' E""]]",n,n
37,v11-2459-0,PANAY,"PANAY, (Géog. mod.)​ île d’Asie, d’environ 100...","[[""10 11' N 137 40' E"", ""10 30' N 139 E""]]","[[""10 11' N 137 40' E"", '30 N 139 E']]",NaN,n
41,v11-325-0,"NÉGREPONT, Isle de","NÉGREPONT, Isle de, (Géog.)​ île de Grèce, app...","[['38 39\' 16"" N 41 32\' E', '38 39\' 16"" N 42...","[[""38 3916' N 41 32' E"", "" N 42 55' E""]]",n,NaN
43,v11-516-0,NICARIA,"NICARIA, (Géog. anc. & mod)​ ou Nicarie ; île ...","[[""37 28' N 43 55' E"", ""37 46' N 44 12' E""]]","[[""37 28' N 43 55' E"", ""46 N 44 12' E""]]",NaN,n
49,v12-1396-11,Pic le,"Pic le, (Geog. mod.)​ autrement le Pic d’Adam,...","[[""5 55' N 98 25' E"", ""5 55' N 98 30' E""]]","[[""5 55' N 98 25' E"", "" N 98 30' E""]]",n,n
